In [ ]:
import cv2
import numpy as np

# Global variables to store clicked points
clicked_points = []

# Mouse callback function to collect four points
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(clicked_points) < 4:
        clicked_points.append([x, y])
        cv2.circle(arch_img_display, (x, y), 5, (0, 255, 0), -1)
        cv2.imshow("Architectural Image - Click 4 points", arch_img_display)

# Load the architectural image (background)
arch_img = cv2.imread('architectural_image.jpg')
arch_img_display = arch_img.copy()

# Load the flag image to be warped and overlaid
flag_img = cv2.imread('flag_image.jpg')
flag_h, flag_w = flag_img.shape[:2]

# Show architectural image and set mouse callback
cv2.imshow("Architectural Image - Click 4 points", arch_img_display)
cv2.setMouseCallback("Architectural Image - Click 4 points", click_event)

print("Please click exactly 4 points on the architectural image in order.")
cv2.waitKey(0)
cv2.destroyAllWindows()

if len(clicked_points) != 4:
    raise ValueError("Exactly 4 points must be selected.")

# Define the source points (corners of the flag image)
pts_flag = np.array([
    [0, 0],
    [flag_w - 1, 0],
    [flag_w - 1, flag_h - 1],
    [0, flag_h - 1]
], dtype=np.float32)

# Convert clicked points to numpy array of float32
pts_arch = np.array(clicked_points, dtype=np.float32)

# Compute homography matrix from flag image corners to clicked points
homography_matrix, status = cv2.findHomography(pts_flag, pts_arch)

# Warp the flag image to the architectural image plane using homography
warped_flag = cv2.warpPerspective(flag_img, homography_matrix, (arch_img.shape[1], arch_img.shape[0]))

# Create a mask from the warped flag to blend
warped_gray = cv2.cvtColor(warped_flag, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(warped_gray, 1, 255, cv2.THRESH_BINARY)

# Invert mask to mask out area on architectural image
mask_inv = cv2.bitwise_not(mask)

# Black-out the area of flag in the architectural image
img_bg = cv2.bitwise_and(arch_img, arch_img, mask=mask_inv)

# Take only region of flag from warped flag image
flag_fg = cv2.bitwise_and(warped_flag, warped_flag, mask=mask)

# Add flag to architectural image
result = cv2.add(img_bg, flag_fg)

# Show the result
cv2.imshow("Flag overlaid on Architectural Image", result)
cv2.waitKey(0)
cv2.destroyAllWindows()


AttributeError: 'NoneType' object has no attribute 'copy'